# Tutorial 14: Custom PyTorch model (`%%krauncher`)

Runs a hand-written CNN training cell on the cheapest suitable remote GPU. The
notebook kernel stays local; only the marked cell goes remote. The model is not
in the HF registry, exercising the analyzer's fallback path.

Config (API key, broker/analyzer URLs) is loaded by the SDK from the sibling
`cas-client/.env` via `CAS_CLIENT_CONFIG` — no secrets in the notebook. The
first `%%krauncher` cell uses `--estimate` to classify and quote only; the
second runs it for real and injects the outputs back as notebook variables.

In [3]:
%env CAS_CLIENT_CONFIG=../../cas-client/.env

env: CAS_CLIENT_CONFIG=../../cas-client/.env


In [4]:
%load_ext krauncher_magic

In [5]:
epochs = 5
batch_size = 32

In [6]:
%%krauncher --in epochs,batch_size --out initial_loss,final_loss,device --pip torch --timeout 300 --estimate
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.bn(self.conv(x))))

class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    n_batches = 50
    for i in range(n_batches):
        images = torch.randn(batch_size, 3, 64, 64, device=device)
        labels = torch.randint(0, 10, (batch_size,), device=device)

        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        print(f"\rEpoch {epoch}/{epochs}  batch {i+1}/{n_batches}  loss={loss.item():.4f}", end="")

    avg = epoch_loss / n_batches
    losses.append(avg)
    print(f"\rEpoch {epoch}/{epochs}  avg_loss={avg:.4f}          ")

initial_loss = round(losses[0], 4)
final_loss = round(losses[-1], 4)
device = str(device)

  CU: iterations from AST loop analysis=250
  CU: CNN classification (FLOPs): ×0.7 × (img=224/224)^0.65=1.0000 → ×0.7000
  CU: num_calls=5, iters_per_call=50 → total_iters=250
  CU: dps=small (data_gb=1.9), cps=small (tflops=0.031)
  CU: weights(cv_training/small/small/small/few/small): gpu=0.2, stor=0.3, cpu=0.4, pcie=0.1, net=0.01, mem=0.0
  CU: pip_baseline: 1s
  CU: t_io_ref = (0 + 0) / 200 + 1 = 1.00s
  CU: K_algo[A](cv_training, small/small/small, few/small)=41.7147
  CU: t_compute_ref = 1.00 × (0.60/0.41) × 41.7147 = 61.05s
  CU: CU = cu_setup(6000) + cu_io(2000) + cu_warmup(0) + cu_compute(122092) = 130092
Classification: tier=light, VRAM=1GB, CU=130091.8 (compute=122091.8, io=2000.0), method=ast, workload=cv_training, model_size=small, working_set=medium, data/step=small(1.9GB), compute/step=small(0.03TF), profile=[ci=0.40,si=0.10,cu=0.30,pcie=0.40,net=0.05], num_calls=5, iters_per_call=50, num_epochs=5, batch_size=32, time=0.52s
estimate_only=true — skipping broker submission

krauncher: estimate — cv_training, ≥1 GB VRAM, 130091.8 CU, ~65s (ref)


In [7]:
%%krauncher --in epochs,batch_size --out initial_loss,final_loss,device --pip torch --timeout 300
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.bn(self.conv(x))))

class ImageClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    n_batches = 50
    for i in range(n_batches):
        images = torch.randn(batch_size, 3, 64, 64, device=device)
        labels = torch.randint(0, 10, (batch_size,), device=device)

        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        print(f"\rEpoch {epoch}/{epochs}  batch {i+1}/{n_batches}  loss={loss.item():.4f}", end="")

    avg = epoch_loss / n_batches
    losses.append(avg)
    print(f"\rEpoch {epoch}/{epochs}  avg_loss={avg:.4f}          ")

initial_loss = round(losses[0], 4)
final_loss = round(losses[-1], 4)
device = str(device)

  CU: iterations from AST loop analysis=250
  CU: CNN classification (FLOPs): ×0.7 × (img=224/224)^0.65=1.0000 → ×0.7000
  CU: num_calls=5, iters_per_call=50 → total_iters=250
  CU: dps=small (data_gb=1.9), cps=small (tflops=0.031)
  CU: weights(cv_training/small/small/small/few/small): gpu=0.2, stor=0.3, cpu=0.4, pcie=0.1, net=0.01, mem=0.0
  CU: pip_baseline: 1s
  CU: t_io_ref = (0 + 0) / 200 + 1 = 1.00s
  CU: K_algo[A](cv_training, small/small/small, few/small)=41.7147
  CU: t_compute_ref = 1.00 × (0.60/0.41) × 41.7147 = 61.05s
  CU: CU = cu_setup(6000) + cu_io(2000) + cu_warmup(0) + cu_compute(122092) = 130092
Classification: tier=light, VRAM=1GB, CU=130091.8 (compute=122091.8, io=2000.0), method=ast, workload=cv_training, model_size=small, working_set=medium, data/step=small(1.9GB), compute/step=small(0.03TF), profile=[ci=0.40,si=0.10,cu=0.30,pcie=0.40,net=0.05], num_calls=5, iters_per_call=50, num_epochs=5, batch_size=32, time=0.52s
Waiting for worker...


krauncher: estimate — cv_training, ≥1 GB VRAM, 130091.8 CU, ~65s (ref)


Host obtained: RTX 5060 Ti, 15GB, (vastai)
Waiting for provision (up to 2 min)
[relay] connecting task_id=c0a8f4d5 target=192.227.212.145:9001 tls=True e2e=True
[relay] TaskStream open task_id=c0a8f4d5
[relay] msg seq=1 type=event data_keys=['name', 'pub']
[relay] key_exchange received task_id=c0a8f4d5
[relay] shared key derived task_id=c0a8f4d5
[relay] uploading payload task_id=c0a8f4d5 plain_len=2445 enc_len=3300
[relay] payload uploaded ok task_id=c0a8f4d5
[relay] msg seq=2 type=event data_keys=['name', 'pub']
[event] key_exchange
[relay] msg seq=3 type=event data_keys=['enc']
[event] execution_started
Wait time: 106 sec
Executing on worker-4998caba: RTX 5060 Ti, 15GB, (vastai), storage 2127 MB/s, PCIe 13.2 GB/s, net 43 Mbps
[relay] msg seq=4 type=metric data_keys=['enc']
[relay] msg seq=5 type=stdout data_keys=['enc']
[relay] msg seq=6 type=stdout data_keys=['enc']
[relay] msg seq=7 type=stdout data_keys=['enc']
[relay] msg seq=8 type=stdout data_keys=['enc']
[relay] msg seq=9 type

krauncher: done on NVIDIA GeForce RTX 5060 Ti in 14.4s — 0.85 KU → initial_loss, final_loss, device


In [8]:
print(f"initial_loss={initial_loss}  final_loss={final_loss}  device={device}")

initial_loss=2.341  final_loss=2.3029  device=cuda
